# Reshaping Data

In this notebook we play around with various reshaping methods available for Series and DataFrames:

- **pivot()** and **pivot_table()**: Group unique values within one or more discrete categories.
- **stack()** and **unstack()**: Pivot a column or row level to the opposite axis respectively.
- **melt()** and **wide_to_long()**: Unpivot a wide DataFrame to a long format.
- **get_dummies()** and **from_dummies()**: Conversions with indicator variables.
- **explode()**: Convert a column of list-like values to individual rows.
- **crosstab()**: Calculate a cross-tabulation of multiple 1 dimensional factor arrays.
- **cut()**: Transform continuous variables to discrete, categorical values
- **factorize()**: Encode 1 dimensional variables into integer labels.


## pivot()

Parameters
The pivot() function typically takes three main arguments: 

- index: The column (or columns) whose unique values will form the new DataFrame''s index (row labels).
- columns: The column (or columns) whose unique values will become the new DataFrame''s column headers.
- values: The column (or columns) to use for populating the new DataFrame''s cell values. If not specified, all remaining columns will be used. 


In [1]:
import numpy as np
import pandas as pd

In [2]:
data = {
    "value": range(12),
    "variable": ["A"] * 3 + ["B"] * 3 + ["C"] * 3 + ["D"] * 3,
    "date": pd.to_datetime(["2020-01-03", "2020-01-04", "2020-01-05"] * 4)
}
df = pd.DataFrame(data)
df

,value,variable,date
0,0,A,2020-01-03
1,1,A,2020-01-04
2,2,A,2020-01-05
3,3,B,2020-01-03
4,4,B,2020-01-04
5,5,B,2020-01-05
6,6,C,2020-01-03
7,7,C,2020-01-04
8,8,C,2020-01-05
9,9,D,2020-01-03


The above dataframe is clearly in LONG format. You can imagine it as some type of time series collection of four variables: A, B, C and D.

To perform time series operations with each unique variable, a better representation would be where the columns are the unique variables and an index of dates identifies individual observations. To reshape the data into this form, we use the DataFrame.pivot() method (also implemented as a top level function pivot()):

In [3]:
pivoted = df.pivot(index="date", columns="variable", values="value")
pivoted

variable,A,B,C,D
date,,,,
2020-01-03,0,3,6,9
2020-01-04,1,4,7,10
2020-01-05,2,5,8,11


# Testing stack() and unstack()

In [4]:
df = pd.DataFrame({
    ("Sales", "Q1"): [100, 120],
    ("Sales", "Q2"): [110, 130],
    ("Costs", "Q1"): [60, 70],
    ("Costs", "Q2"): [65, 75],
}, index=["Product A", "Product B"])

df.columns = pd.MultiIndex.from_tuples(df.columns)
df

Sales      Costs    
             Q1   Q2    Q1  Q2
Product A   100  110    60  65
Product B   120  130    70  75

In [5]:
# view index
df.index

Index(['Product A', 'Product B'], dtype='object')

In [6]:
df.columns

MultiIndex([('Sales', 'Q1'),
            ('Sales', 'Q2'),
            ('Costs', 'Q1'),
            ('Costs', 'Q2')],
           )

In [7]:
df.columns.names

FrozenList([None, None])

In [8]:
df.columns.levels

FrozenList([['Costs', 'Sales'], ['Q1', 'Q2']])

In [9]:
df['Sales']

,Q1,Q2
Product A,100,110
Product B,120,130


In [12]:
df_stack = df.stack(future_stack=True)
df_stack

Sales  Costs
Product A Q1    100     60
          Q2    110     65
Product B Q1    120     70
          Q2    130     75

In [13]:
df_stack.index

MultiIndex([('Product A', 'Q1'),
            ('Product A', 'Q2'),
            ('Product B', 'Q1'),
            ('Product B', 'Q2')],
           )

In [15]:
df_stack_2 = df.stack(future_stack=True, level=0)
df_stack_2

Q1   Q2
Product A Sales  100  110
          Costs   60   65
Product B Sales  120  130
          Costs   70   75

In [16]:
arrays = [
    ["East", "East", "West", "West"],
    ["Store A", "Store B", "Store A", "Store B"]
]

index = pd.MultiIndex.from_arrays(arrays, names=["Region", "Store"])

df2 = pd.DataFrame({
    "Sales": [1200, 900, 1500, 1300],
    "Employees": [10, 7, 12, 9]
}, index=index)

df2

Sales  Employees
Region Store                    
East   Store A   1200         10
       Store B    900          7
West   Store A   1500         12
       Store B   1300          9

In [17]:
df2.index

MultiIndex([('East', 'Store A'),
            ('East', 'Store B'),
            ('West', 'Store A'),
            ('West', 'Store B')],
           names=['Region', 'Store'])

In [18]:
df2.columns

Index(['Sales', 'Employees'], dtype='object')

In [19]:
df2_stack = df2.stack()
df2_stack

Region  Store             
East    Store A  Sales        1200
                 Employees      10
        Store B  Sales         900
                 Employees       7
West    Store A  Sales        1500
                 Employees      12
        Store B  Sales        1300
                 Employees       9
dtype: int64

In [20]:
df2_stack.index

MultiIndex([('East', 'Store A',     'Sales'),
            ('East', 'Store A', 'Employees'),
            ('East', 'Store B',     'Sales'),
            ('East', 'Store B', 'Employees'),
            ('West', 'Store A',     'Sales'),
            ('West', 'Store A', 'Employees'),
            ('West', 'Store B',     'Sales'),
            ('West', 'Store B', 'Employees')],
           names=['Region', 'Store', None])

In [21]:
df2

Sales  Employees
Region Store                    
East   Store A   1200         10
       Store B    900          7
West   Store A   1500         12
       Store B   1300          9

In [22]:
df2.unstack()

Sales         Employees        
Store  Store A Store B   Store A Store B
Region                                  
East      1200     900        10       7
West      1500    1300        12       9

In [23]:
df2.unstack(level=0)

Sales       Employees     
Region   East  West      East West
Store                             
Store A  1200  1500        10   12
Store B   900  1300         7    9

In [24]:
df

Sales      Costs    
             Q1   Q2    Q1  Q2
Product A   100  110    60  65
Product B   120  130    70  75

In [28]:
df.stack(future_stack=True).stack(future_stack=True)

Product A  Q1  Sales    100
               Costs     60
           Q2  Sales    110
               Costs     65
Product B  Q1  Sales    120
               Costs     70
           Q2  Sales    130
               Costs     75
dtype: int64

In [29]:
df

Sales      Costs    
             Q1   Q2    Q1  Q2
Product A   100  110    60  65
Product B   120  130    70  75

In [30]:
df.stack(future_stack=True).stack(future_stack=True).unstack().unstack()

Sales      Costs    
             Q1   Q2    Q1  Q2
Product A   100  110    60  65
Product B   120  130    70  75

## swaplevel() - the index-level shuffler

In [31]:
import pandas as pd

arrays = [
    ["East", "East", "West", "West"],
    ["Store A", "Store B", "Store A", "Store B"]
]

index = pd.MultiIndex.from_arrays(arrays, names=["Region", "Store"])

df = pd.DataFrame({"Sales": [1200, 900, 1500, 1300]}, index=index)
df

Sales
Region Store         
East   Store A   1200
       Store B    900
West   Store A   1500
       Store B   1300

In [32]:
df.swaplevel()

,,Sales
Store,Region,
Store A,East,1200
Store B,East,900
Store A,West,1500
Store B,West,1300


In [33]:
df2 = pd.DataFrame({
    ("Sales", "Q1"): [100, 120],
    ("Sales", "Q2"): [110, 130],
    ("Costs", "Q1"): [60, 70],
    ("Costs", "Q2"): [65, 75],
}, index=["A", "B"])

df2.columns = pd.MultiIndex.from_tuples(df2.columns)
df2

Sales      Costs    
     Q1   Q2    Q1  Q2
A   100  110    60  65
B   120  130    70  75

In [34]:
df2.swaplevel(axis=1)

,Q1,Q2,Q1,Q2
,Sales,Sales,Costs,Costs
A,100,110,60,65
B,120,130,70,75


In [36]:
df

Sales      Costs    
     Q1   Q2    Q1  Q2
A   100  110    60  65
B   120  130    70  75

In [37]:
df3 = df.swaplevel()
df3

,,Sales
Store,Region,
Store A,East,1200
Store B,East,900
Store A,West,1500
Store B,West,1300


In [38]:
df3.sort_index()

Sales
Store   Region       
Store A East     1200
        West     1500
Store B East      900
        West     1300

In [39]:
df3.sort_index(level=["Region", "Store"])

,,Sales
Store,Region,
Store A,East,1200
Store B,East,900
Store A,West,1500
Store B,West,1300


In [41]:
df = pd.DataFrame({
    ("Sales", "Q1"): [100, 120],
    ("Sales", "Q2"): [110, 130],
    ("Costs", "Q1"): [60, 70],
    ("Costs", "Q2"): [65, 75],
}, index=["Product A", "Product B"])

df.columns = pd.MultiIndex.from_tuples(df.columns)
df

Sales      Costs    
             Q1   Q2    Q1  Q2
Product A   100  110    60  65
Product B   120  130    70  75

In [43]:
s = df.stack(future_stack=True)
s

Sales  Costs
Product A Q1    100     60
          Q2    110     65
Product B Q1    120     70
          Q2    130     75

In [44]:
s2 = s.swaplevel()
s2

,,Sales,Costs
Q1,Product A,100,60
Q2,Product A,110,65
Q1,Product B,120,70
Q2,Product B,130,75


In [45]:
s2.sort_index()

Sales  Costs
Q1 Product A    100     60
   Product B    120     70
Q2 Product A    110     65
   Product B    130     75